In [1]:
import pandas as pd
import json

Goal: filter templates such that they exactly match the wikitext

In [21]:
df_orig = pd.read_csv('etytree_large_joined.csv')

In [25]:
df = pd.read_csv('etytree_large_joined.csv')

In [75]:
import ast

stdf = df_orig['templates'].apply(ast.literal_eval)

In [80]:
df['templates'] = stdf

In [121]:
def is_subseq(smaller, larger, debugme=False):
    it = iter(larger)
    result = []
    # return all(any(c == ch for c in it) for ch in smaller)
    for ch in smaller:
        for c in it:
            if c == ch:
                if debugme:
                    print(f"Matched {c} with {ch}")
                result.append(c) # build from larger sequence
                break
        else:
            if debugme:
                print(f"Failed to find {ch} in {larger}")
            return None
    return result
# def is_subseq_v2(smaller, larger):
    # return all((ch in larger) for ch in smaller)

In [104]:
is_subseq('mali', 'maila', debugme=True), is_subseq('mali', 'maileeei') # False
# is_subseq_v2('mali', 'maila') # True

Failed to find i in maila


(None, ['m', 'a', 'l', 'i'])

In [123]:
import importlib
import notebooks.clean_templates_util as clean_templates_util
importlib.reload(clean_templates_util)
from notebooks.clean_templates_util import is_ignored_template, ReducedTemplate, reconstruct_template, process_recursive_templates

def match_templates(wikitext, wtp_templates, debugme=False):    
    # filter the wikitext templates to be ones that are top-level, so they are found in the wikitext
    # reconstructed = [(template, reconstruct_template(template)) for template in wtp_templates]
    reconstructed = [ReducedTemplate.from_dict(template) for template in wtp_templates]
    
    # extract templates from wikitext
    textual = []
    
    # note: make it also work for recursive templates like {{m|en|hello {{m|en|world}}}}
    
    # for template in wikitext.split('{{')[1:]:
    #     x = template.split('}}')[0]
    #     textual.append(ReducedTemplate.from_wikitext(x))
    textual = process_recursive_templates(wikitext)
    textual = [x for x in textual if not is_ignored_template(x.name)]

    
    # print(reconstructed)
    # remove [[ and ]] from wtx_extracted, because they are links
    
    # print(textual)
    
    # textual should be a subsequence of reconstructed
    # filter reconstructed to be those that are also in textual
    filtered = is_subseq(textual, reconstructed, debugme)
    
    return filtered
def _pd_obtain_filtered_templates(wikitext, wtp_templates, debugme=False):
    out = match_templates(wikitext, wtp_templates, debugme)
    if out is None:
        return None
    return [x.original_dict for x in out]

In [118]:
(result := match_templates(df.loc[0, 'ety_text'], df.loc[0, 'templates'])), df.loc[0, 'ety_text'].count('{{'), len(df.loc[0, 'templates'])

([{{root|en|ine-pro|*deyḱ-}},
  {{inh|en|enm|dixionare}},
  {{glossary|learned borrowing}},
  {{der|en|ML.|dictiōnārium}},
  {{der|en|la|dictiōnārius}},
  {{m|la|dictiō||speaking}},
  {{m|la|dictus}},
  {{m|la|dīcō|t=speak}},
  {{m|la|-ārium|t=room, place}},
  {{surf|en|diction|-ary}}],
 10,
 10)

In [119]:


_pd_obtain_filtered_templates(df.loc[0, 'ety_text'], df.loc[0, 'templates'])

[{'name': 'root',
  'args': {'1': 'en', '2': 'ine-pro', '3': '*deyḱ-'},
  'expansion': ''},
 {'name': 'inh',
  'args': {'1': 'en', '2': 'enm', '3': 'dixionare'},
  'expansion': 'Middle English dixionare'},
 {'name': 'glossary',
  'args': {'1': 'learned borrowing'},
  'expansion': 'learned borrowing'},
 {'name': 'der',
  'args': {'1': 'en', '2': 'ML.', '3': 'dictiōnārium'},
  'expansion': 'Medieval Latin dictiōnārium'},
 {'name': 'der',
  'args': {'1': 'en', '2': 'la', '3': 'dictiōnārius'},
  'expansion': 'Latin dictiōnārius'},
 {'name': 'm',
  'args': {'1': 'la', '2': 'dictiō', '3': '', '4': 'speaking'},
  'expansion': 'dictiō (“speaking”)'},
 {'name': 'm', 'args': {'1': 'la', '2': 'dictus'}, 'expansion': 'dictus'},
 {'name': 'm',
  'args': {'1': 'la', '2': 'dīcō', 't': 'speak'},
  'expansion': 'dīcō (“speak”)'},
 {'name': 'm',
  'args': {'1': 'la', '2': '-ārium', 't': 'room, place'},
  'expansion': '-ārium (“room, place”)'},
 {'name': 'surf',
  'args': {'1': 'en', '2': 'diction', '3':

In [130]:
# apply to the df
df['filtered'] = df.apply(lambda x: _pd_obtain_filtered_templates(x['ety_text'], x['templates']), axis=1)

In [100]:
len(df)

21376

In [131]:
# obtain rows of df that are None
nully = df[df['filtered'].isnull()]
nully

# 2913 rows are None (v1)
# 2158 rows are None (v2)
# 1935 rows are None (v3)

# note: a known problem is recursive templates like {{m|en|hello {{m|en|hello}} }}
# fixed: 930 rows are None (v4)
# 700 are None (v5)

,word,language,ety_number,ety,ety_text,templates,num_templates,filtered
10,Tuesday,English,0,"From Middle English Tewesday, from Old English...",{{root|en|ine-pro|*dʰegʷʰ-}} From {{inh|en|enm...,"[{'name': 'root', 'args': {'1': 'en', '2': 'in...",15,None
23,abacus,English,0,"From Late Middle English abacus, abagus, agabu...",{{multiple images |direction = vertical |image...,"[{'name': 'nb...', 'args': {'1': 'Argen􂀿toratu...",15,None
43,車,Korean,1,From Middle Chinese 車 (MC tsyhae).,From {{der|ko|ltc|sort=차|-}} {{ltc-l|車|id=2}}....,"[{'name': 'der', 'args': {'1': 'ko', '2': 'ltc...",10,None
44,車,Korean,2,From Middle Chinese 車 (MC kjo).,From {{der|ko|ltc|sort=거|-}} {{ltc-l|車|id=1}}....,"[{'name': 'der', 'args': {'1': 'ko', '2': 'ltc...",11,None
84,taghairm,English,0,"Borrowed from Scottish Gaelic taghairm, from O...",{{root|en|ine-pro|*ǵeh₂r-}} Borrowed from {{bo...,"[{'name': 'root', 'args': {'1': 'en', '2': 'in...",13,None
...,...,...,...,...,...,...,...,...
21269,সাহা,Bengali,0,"Inherited from Magadhi Prakrit 𑀰𑀸𑀳𑀼 (śāhu), wh...",Inherited from [[w:Magadhi Prakrit|Magadhi Pra...,"[{'name': 'm', 'args': {'1': 'inc-ash', '2': '...",13,None
21324,つぶし,Japanese,2,First cited to a manuscript of the Ruijū Myōgi...,{{ja-kanjitab|alt=腿}} First cited to a manuscr...,"[{'name': 'cog', 'args': {'1': 'jpx-ryu-pro', ...",11,None
21336,قانی قاینامق,Ottoman Turkish,0,"From قان (kan, “blood”) + ـی (-ı, possessive s...",From {{com|ota|قان|tr1=kan|t1=blood|ـی|tr2=-ı|...,"[{'name': 'com', 'args': {'1': 'ota', '2': 'قا...",12,None
21356,হল,Bengali,2,Borrowed from Sanskrit হল (hala). Compare Odia...,[[File:Bengali Farmer with Plow and Yoke on hi...,"[{'name': 'lang', 'args': {'1': 'bn', '2': 'কা...",10,None


In [132]:
# save to csv
nully.to_csv('unmatched_wikitexts_v5.csv', index=False)

In [133]:
# drop rows of df that are None
# original: 21376 rows

df_good = df.dropna(subset=['filtered'])
print(len(df_good)) # 15922
# 20676: 96.7% is good enough


20676


In [136]:
# drop rows of df such that len(templates) and (num of "{{" in string) are different
# allow "<=" because for instance wtp removes "cite-" and certain non-ety templates
df_clean = df_good[df_good['filtered'].apply(len) <= df_good['ety_text'].apply(lambda x: x.count('{{'))]
df_clean = df_clean.reset_index(drop=True) # 15922
# 16520
# 20676

In [137]:
df_clean

,word,language,ety_number,ety,ety_text,templates,num_templates,filtered
0,dictionary,English,0,"From Middle English dixionare, learned borrowi...",{{root|en|ine-pro|*deyḱ-}} From {{inh|en|enm|d...,"[{'name': 'root', 'args': {'1': 'en', '2': 'in...",10,"[{'name': 'root', 'args': {'1': 'en', '2': 'in..."
1,pound,English,1,"From Middle English pound, from Old English pu...",{{root|en|ine-pro|*(s)pend-}} From {{inh|en|en...,"[{'name': 'root', 'args': {'1': 'en', '2': 'in...",13,"[{'name': 'root', 'args': {'1': 'en', '2': 'in..."
2,pie,English,1,"From Middle English pye, pie, pey, perhaps fro...","From {{inh|en|enm|pye}}, {{m|enm|pie}}, {{m|en...","[{'name': 'inh', 'args': {'1': 'en', '2': 'enm...",14,"[{'name': 'inh', 'args': {'1': 'en', '2': 'enm..."
3,pie,Spanish,1,"Inherited from Old Spanish pie, from Latin ped...",{{dercat|es|itc-pro|ine-pro|inh=2}} {{inh+|es|...,"[{'name': 'dercat', 'args': {'1': 'es', '2': '...",10,"[{'name': 'dercat', 'args': {'1': 'es', '2': '..."
4,A,English,1,From Middle English and Old English upper case...,From {{der|en|enm|-}} and {{der|en|ang|-}} upp...,"[{'name': 'der', 'args': {'1': 'en', '2': 'enm...",12,"[{'name': 'der', 'args': {'1': 'en', '2': 'enm..."
...,...,...,...,...,...,...,...,...
20671,teich,Scottish Gaelic,0,"From Old Irish teichid (Irish teith, Manx çhea...",{{root|gd|ine-pro|*tekʷ-}} From {{inh|gd|sga|t...,"[{'name': 'root', 'args': {'1': 'gd', '2': 'in...",11,"[{'name': 'root', 'args': {'1': 'gd', '2': 'in..."
20672,পদ,Bengali,0,"Inherited from Sanskrit পদ (pada, “step, foot,...","{{inh+|bn|sa|पद|পদ |step, foot, position}}, fr...","[{'name': 'glossary', 'args': {'1': 'Inherited...",14,"[{'name': 'inh+', 'args': {'1': 'bn', '2': 'sa..."
20673,নীতি,Bengali,0,Borrowed from Sanskrit নীতি (nīti). Cognate wi...,{{bor+|bn|sa|नीति|নীতি}}. Cognate with {{cog|a...,"[{'name': 'glossary', 'args': {'1': 'loanword'...",10,"[{'name': 'bor+', 'args': {'1': 'bn', '2': 'sa..."
20674,জগৎ,Bengali,0,Borrowed from Sanskrit জগৎ (jagat). Cognate wi...,{{bor+|bn|sa|जगत्|জগৎ}}. Cognate with {{cog|as...,"[{'name': 'glossary', 'args': {'1': 'loanword'...",10,"[{'name': 'bor+', 'args': {'1': 'bn', '2': 'sa..."


In [138]:
df_clean.to_csv('etytree_large_joined_clean_v2.csv', index=False)

Inspect

In [124]:
_pd_obtain_filtered_templates(df.loc[10, 'ety_text'], df.loc[10, 'templates'], debugme=True)

Matched {{root|en|ine-pro|*dʰegʷʰ-}} with {{root|en|ine-pro|*dʰegʷʰ-}}
Matched {{inh|en|enm|Tewesday}} with {{inh|en|enm|Tewesday}}
Matched {{inh|en|ang|tīwesdæġ||Tuesday}} with {{inh|en|ang|tīwesdæġ||Tuesday}}
Matched {{inh|en|gmw-pro|*Tīwas dag||Tuesday|lit=Tiw's Day}} with {{inh|en|gmw-pro|*Tīwas dag||Tuesday|lit=Tiw's Day}}
Matched {{cog|la|diēs Mārtis}} with {{cog|la|diēs Mārtis}}
Matched {{cog|grc|Ἄρεως ἡμέρα}} with {{cog|grc|Ἄρεως ἡμέρα}}
Matched {{cog|sco|Tysday||Tuesday}} with {{cog|sco|Tysday||Tuesday}}
Matched {{cog|stq|Täisdai||Tuesday}} with {{cog|stq|Täisdai||Tuesday}}
Matched {{cog|fy|tiisdei||Tuesday}} with {{cog|fy|tiisdei||Tuesday}}
Matched {{cog|de|Ziestag||Tuesday}} with {{cog|de|Ziestag||Tuesday}}
Matched {{cog|da|tirsdag||Tuesday}} with {{cog|da|tirsdag||Tuesday}}
Matched {{cog|sv|tisdag||Tuesday}} with {{cog|sv|tisdag||Tuesday}}
Failed to find {{cog|fi|tiistai||Tuesday}} in [{{root|en|ine-pro|*dʰegʷʰ-}}, {{inh|en|enm|Tewesday}}, {{inh|en|ang|tīwesdæġ||Tuesday}}, 

In [128]:
a = ReducedTemplate.from_wikitext("{{cog|fi|tiistai||Tuesday}}")
b = ReducedTemplate.from_wikitext("{{cog|fy|tiistai||Tuesday}}")
a in [None, b]

True

In [113]:
df.loc[10, 'ety_text'], df.loc[10, 'templates']

("{{root|en|ine-pro|*dʰegʷʰ-}} From {{inh|en|enm|Tewesday}}, from {{inh|en|ang|tīwesdæġ||Tuesday}}, from {{inh|en|gmw-pro|*Tīwas dag||Tuesday|lit=Tiw's Day}}.  This was a [[interpretatio germanica|Germanic interpretation]] of {{cog|la|diēs Mārtis}}, itself a translation of {{cog|grc|Ἄρεως ἡμέρα}} ({{w|interpretatio romana}}). Cognate with {{cog|sco|Tysday||Tuesday}}, {{cog|stq|Täisdai||Tuesday}}, {{cog|fy|tiisdei||Tuesday}}, dialectal {{cog|de|Ziestag||Tuesday}}, {{cog|da|tirsdag||Tuesday}}, {{cog|sv|tisdag||Tuesday}}, {{cog|fi|tiistai||Tuesday}}. More at {{m|en|Tyr}}, {{m|en|day}}. {{root|en|ine-pro|*dyew-}} ",
 [{'name': 'root',
   'args': {'1': 'en', '2': 'ine-pro', '3': '*dʰegʷʰ-'},
   'expansion': ''},
  {'name': 'inh',
   'args': {'1': 'en', '2': 'enm', '3': 'Tewesday'},
   'expansion': 'Middle English Tewesday'},
  {'name': 'inh',
   'args': {'1': 'en', '2': 'ang', '3': 'tīwesdæġ', '4': '', '5': 'Tuesday'},
   'expansion': 'Old English tīwesdæġ (“Tuesday”)'},
  {'name': 'inh',
 

Testing

In [94]:
print(process_recursive_templates("{{m|en|hello {{m|en|world}}}}"))
print(process_recursive_templates("{{m|en|hello}} {{m|en|world}} {{m|en|foo{{m|en|bar}}|lish}}"))

[{{men|world}}, {{men|hello }}]
[{{men|hello}}, {{men|world}}, {{men|bar}}, {{men|foo|lish}}]
